# Final Training & Deployment Pipeline

This notebook trains the final XGBoost credit risk model, builds a unified preprocessing + modeling pipeline, and exports a single deployable artifact: `credit_risk_pipeline.pkl`.

**Run this notebook end-to-end to generate the final model file.**

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import joblib

## 2. Load Dataset

In [ ]:
df = pd.read_csv('data/credit_risk.csv')
df.head()

## 3. Define Features & Target

In [ ]:
target = 'loan_status'

num_cols = [
    'person_age','person_income','person_emp_length',
    'loan_amnt','loan_int_rate','loan_percent_income',
    'cb_person_cred_hist_length'
]

cat_cols = [
    'person_home_ownership','loan_intent',
    'loan_grade','cb_person_default_on_file'
]

X = df[num_cols + cat_cols]
y = df[target]

## 4. Train-Test Split (Leakage Safe)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## 5. Preprocessing Pipelines

In [ ]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

grade_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[['A','B','C','D','E','F','G']]))
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('grade', grade_pipeline, ['loan_grade']),
    ('cat', cat_pipeline, list(set(cat_cols) - {'loan_grade'}))
])

## 6. Train XGBoost Model

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb_model)
])

pipeline.fit(X_train, y_train)

## 7. Evaluation

In [ ]:
train_auc = roc_auc_score(y_train, pipeline.predict_proba(X_train)[:,1])
test_auc = roc_auc_score(y_test, pipeline.predict_proba(X_test)[:,1])

train_auc, test_auc

## 8. Save Final Deployment Pipeline

In [ ]:
joblib.dump(pipeline, 'models/credit_risk_pipeline.pkl')

## ✅ Final Output

The file `models/credit_risk_pipeline.pkl` is now ready for:
- Streamlit deployment
- SHAP explainability
- Reproducible inference

**This is the ONLY model file that should be committed to GitHub.**